In [2]:
import platform
import subprocess

# 1. Print the operating system details
print("--- Operating System ---")
print(f"OS: {platform.system()} {platform.release()}")
print(f"Processor: {platform.processor()}")

# 2. Check if the server is hosted by Google (Colab)
print("\n--- Provider Verification ---")
try:
    hostname = subprocess.check_output("hostname", shell=True).decode().strip()
    print(f"Server Hostname: {hostname}")
    if "colab" in hostname or "google" in hostname.lower():
        print("✅ Success: You are running on a Google Colab server!")
    else:
        print("❌ Notice: This does not look like a Google server.")
except Exception as e:
    print(f"Could not fetch hostname: {e}")

# 3. Check for hardware accelerators (GPU)
print("\n--- Hardware Check ---")
try:
    gpu_info = subprocess.check_output("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True).decode().strip()
    print(f"GPU Available: {gpu_info}")
except Exception:
    print("GPU Available: None (Running on standard cloud CPU)")


--- Operating System ---
OS: Linux 6.6.122+
Processor: x86_64

--- Provider Verification ---
Server Hostname: 70b5c42f7c02
❌ Notice: This does not look like a Google server.

--- Hardware Check ---
GPU Available: None (Running on standard cloud CPU)


In [ ]:
import cupy as cp
import cudf
import time

print("⚡ Initializing 10-Million-Row GPU Dataset...")
start_time = time.time()

# 1. Generate 10 million rows of synthetic flight data directly on the GPU
n_rows = 10_000_000
cp.random.seed(42)

# Generate high-cardinality categorical data and continuous delays
carriers = [b'Delta', b'United', b'American', b'Southwest', b'JetBlue', b'Spirit', b'Alaska']
airports = [b'JFK', b'LAX', b'ORD', b'DFW', b'DEN', b'SFO', b'ATL', b'MIA', b'SEA', b'BOS']

df_gpu = cudf.DataFrame({
    'Carrier': cp.random.choice(carriers, size=n_rows),
    'Origin': cp.random.choice(airports, size=n_rows),
    'Dest': cp.random.choice(airports, size=n_rows),
    'Distance': cp.random.randint(100, 3000, size=n_rows),
    'DepDelay': cp.random.normal(loc=15, scale=45, size=n_rows).astype('float32'),
    'ArrDelay': cp.random.normal(loc=12, scale=50, size=n_rows).astype('float32'),
    'AirTime': cp.random.randint(30, 360, size=n_rows)
})

# Filter out physically impossible negative air times / delays for cleaner data
df_gpu['DepDelay'] = df_gpu['DepDelay'].clip(lower=-20)
df_gpu['ArrDelay'] = df_gpu['ArrDelay'].clip(lower=-20)

gen_time = time.time() - start_time
print(f"✅ Generated {n_rows:,} rows directly in GPU memory in {gen_time:.2f} seconds.\n")


# 2. HEAVY COMPUTATION 1: Multi-column conditional feature engineering
print("🚀 Computing complex engineered features on GPU...")
comp1_start = time.time()

# Calculate a non-linear "Delay Severity Index" using conditional logic
df_gpu['IsLate'] = (df_gpu['ArrDelay'] > 15).astype('int32')
df_gpu['SeverityIndex'] = (
    (df_gpu['DepDelay'] * 1.5) + 
    (df_gpu['ArrDelay'] * 2.0) - 
    (df_gpu['Distance'] / 500)
) * df_gpu['IsLate']

# Apply a math operation over the 10 million rows
df_gpu['LogDistance'] = cp.log1p(df_gpu['Distance'])

comp1_time = time.time() - comp1_start
print(f"✅ Heavy feature engineering took: {comp1_time:.4f} seconds.")


# 3. HEAVY COMPUTATION 2: Complex multi-key multi-aggregation grouping
print("🚀 Grouping, sorting, and aggregating 10,000,000 records...")
comp2_start = time.time()

summary_gpu = df_gpu.groupby(['Carrier', 'Origin']).agg({
    'DepDelay': ['mean', 'std', 'max'],
    'ArrDelay': ['mean', 'min', 'max'],
    'SeverityIndex': 'mean',
    'IsLate': 'sum'
})

# Flatten multi-index columns for readability
summary_gpu.columns = ['_'.join(col) for col in summary_gpu.columns]
summary_gpu = summary_gpu.sort_values(by='SeverityIndex_mean', ascending=False)

comp2_time = time.time() - comp2_start
print(f"✅ Complex aggregation took: {comp2_time:.4f} seconds.\n")


# 4. Pull result summary to display
print("--- TOP 5 MOST DELAYED FLIGHT ROUTES (GPU RESULT) ---")
print(summary_gpu.head(5))
